# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sah-aditya/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane 2: Refresh / Content Opportunity Scoring.

I am choosing this lane because the decision is operational: when an editor only has time for a small batch of pages, I want to rank pages by where a refresh is most likely to matter. The starter data already has the signals this lane needs — visibility, freshness, CTR, position, and trend direction — so I can build a useful queue before I touch the full warehouse release.

In [5]:
import os
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != os.path.abspath(os.sep):
    os.chdir("..")

print("Working dir:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

rows = len(df)
declining = int(df["is_declining_label"].sum())
declining_rate = df["is_declining_label"].mean()

print(f"Rows: {rows}")
print(f"Declining pages: {declining} ({declining_rate:.1%})")

Working dir: a:\flyrank-ml-internship
Rows: 30000
Declining pages: 16262 (54.2%)


## 2. The question: decision, action, cost of a wrong call

The decision I want to improve is which pages a content strategist should review first.

The action is simple: the strategist takes the top-ranked pages and updates the title, intro, structure, internal links, or other content signals. A wrong recommendation costs time. A false positive wastes an editor's effort on a page that did not need attention, while a false negative lets a declining page keep slipping. A plain rule is not enough because the strongest candidates are not determined by one threshold alone.

In [6]:
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
low_ctr_visible = ((df["impressions_90d"] >= 500) & (df["avg_position"].between(1, 20)) & (df["ctr"] < 0.5)).sum()
page_one_decay = ((df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)).sum()

print(f"Stale + visible pages: {stale_visible}")
print(f"Low-CTR visible pages: {low_ctr_visible}")
print(f"Page-one decay risk pages: {page_one_decay}")

Stale + visible pages: 17
Low-CTR visible pages: 9745
Page-one decay risk pages: 7076


## 3. Quick look at the data (2-3 real numbers)

The starter slice already shows enough signal to justify a refresh queue: 30,000 pages total, 16,262 declining pages, and thousands of pages that are visible but stale or low-CTR. That is enough evidence to test a ranked recommendation flow before I move to the full warehouse release.

In [7]:
visible_stale = (df["impressions_90d"] >= 500) & (df["days_since_last_update"] >= 180)

print(f"Total pages: {len(df)}")
print(f"Declining pages: {declining} ({declining_rate:.1%})")
print(f"Visible + stale pages: {int(visible_stale.sum())}")
print(f"Median CTR among visible + stale pages: {df.loc[visible_stale, 'ctr'].median():.2f}")

Total pages: 30000
Declining pages: 16262 (54.2%)
Visible + stale pages: 17
Median CTR among visible + stale pages: 0.20


## 4. Careful words: what I can and can't claim

I can claim that the starter data supports a ranked refresh queue and that pages with low CTR, stale updates, or weak positions look worth review.

I cannot claim causal impact, search-engine intent, or that a model 'predicts Google.' My output will stay decision-support: an observed ranking of pages to inspect first.

In [8]:
print("Allowed claim style: observed / measured / directional / decision-support")
print("Not allowed: causal proof, 'predicting Google', or uncited client-specific claims")

Allowed claim style: observed / measured / directional / decision-support
Not allowed: causal proof, 'predicting Google', or uncited client-specific claims


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.